# Reinforcement Layer Types with Catalog Integration

This notebook demonstrates the three reinforcement layer types:

1. **BarReinforcement** - Proxy to bar catalog components (steel, carbon)
2. **LayerReinforcement** - Proxy to textile/mat catalog components
3. **AreaReinforcement** - Product-independent (explicit area + material)

## Key Concepts:

- **Product-based workflow** (analysis): Known components → calculate capacity
- **Design-oriented workflow** (synthesis): Required area → optimize → select
- **Proxy pattern**: Layers access catalog without duplication
- **Cached catalogs**: Automatic JSON-based caching for performance (NEW)

## 2. Import Libraries

All catalog functions now use caching automatically.

In [ ]:
import time

# Import catalog manager
from bmcs_cross_section.cs_components import get_catalog_manager

# Get catalog manager instance
manager = get_catalog_manager()

print("Catalog Manager Initialized")
print("=" * 60)

# Demonstrate caching performance
print("\nTesting catalog caching...")

# First call: may create or load from disk
start = time.time()
steel_catalog = manager.get_steel_catalog()
time_first = (time.time() - start) * 1000

# Second call: loads from memory (very fast)
start = time.time()
steel_catalog_2 = manager.get_steel_catalog()
time_second = (time.time() - start) * 1000

print(f"  1st call: {time_first:.2f} ms (create/load from disk)")
print(f"  2nd call: {time_second:.2f} ms (from memory)")
if time_first > 0:
    print(f"  Speedup: {time_first/time_second:.1f}x faster")

# Show cache status
cache_info = manager.get_cache_info()
print(f"\nCache Status:")
print(f"  Memory cached: {cache_info['memory_cached']}")
print(f"  Disk cached: {cache_info['disk_cached']}")
print(f"\n✓ Caching system active")

## 1. Catalog Caching System

**NEW:** Component catalogs are now automatically cached for performance.

### Features:
- **Persistent storage**: Catalogs saved in JSON format (`.catalog_cache/`)
- **Lazy loading**: Catalogs loaded only when needed
- **Memory cache**: Fast access after first load
- **Automatic**: No code changes needed - caching is default behavior

### Benefits:
- ✓ 2-10x faster catalog access after first load
- ✓ Persistent across sessions
- ✓ Transparent - existing code works unchanged

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Component catalog (now with automatic caching)
from bmcs_cross_section.cs_components import (
    create_steel_rebar_catalog,  # Cached by default
    create_concrete_catalog,      # Cached by default
    get_concrete_by_class,        # Uses cached catalog
    SteelRebarComponent,
    CarbonBarComponent,
    TextileReinforcementComponent,
)

# Cross-section design
from bmcs_cross_section.cs_design import (
    RectangularShape,
    TShape,
    BarReinforcement,
    LayerReinforcement,
    AreaReinforcement,
    ReinforcementLayout,  # NEW: Needed to wrap layers
    CrossSection,
)

# Material models
from bmcs_cross_section.matmod import create_steel

plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print("✓ All catalog functions use automatic caching")

## 3. BarReinforcement - Discrete Bar Proxy

**Usage:** Product-based analysis with known bar components.

**Pattern:**
- Link to catalog component
- Specify count
- `A_s` calculated automatically: `component.area × count`

**Note:** Component creation uses cached catalog data automatically.

### Example 3.1: Steel Bars (Cached Components)

In [ ]:
# Select catalog component
steel_d20 = SteelRebarComponent(nominal_diameter=20, grade='B500B')
steel_d16 = SteelRebarComponent(nominal_diameter=16, grade='B500B')

print("Catalog Components:")
print("=" * 60)
print(f"\n{steel_d20.name}")
print(f"  Product ID: {steel_d20.product_id}")
print(f"  Diameter: {steel_d20.nominal_diameter} mm")
print(f"  Area per bar: {steel_d20.area:.2f} mm²")
print(f"  f_tk: {steel_d20.f_tk} MPa")
print(f"  f_td: {steel_d20.f_td:.1f} MPa (design)")

print(f"\n{steel_d16.name}")
print(f"  Product ID: {steel_d16.product_id}")
print(f"  Area per bar: {steel_d16.area:.2f} mm²")

In [ ]:
# Create BarReinforcement layers (proxies)
bottom_layer = BarReinforcement(
    z=450,
    component=steel_d20,  # Link to catalog
    count=6,               # Number of bars
    name='Bottom'
)

top_layer = BarReinforcement(
    z=50,
    component=steel_d16,   # Different component
    count=4,
    name='Top'
)

print("BarReinforcement Layers:")
print("=" * 60)
print(f"\nBottom layer:")
print(f"  Position: z = {bottom_layer.z} mm")
print(f"  Component: {bottom_layer.component.product_id}")
print(f"  Count: {bottom_layer.count} bars")
print(f"  Diameter: {bottom_layer.diameter} mm")
print(f"  A_s = {bottom_layer.count} × {bottom_layer.component.area:.2f} = {bottom_layer.A_s:.2f} mm²")

print(f"\nTop layer:")
print(f"  Position: z = {top_layer.z} mm")
print(f"  Component: {top_layer.component.product_id}")
print(f"  Count: {top_layer.count} bars")
print(f"  Diameter: {top_layer.diameter} mm")
print(f"  A_s = {top_layer.count} × {top_layer.component.area:.2f} = {top_layer.A_s:.2f} mm²")

print(f"\nTotal reinforcement: {bottom_layer.A_s + top_layer.A_s:.2f} mm²")

### Example 3.2: Cross-Section with BarReinforcement

Concrete component loaded from cached catalog.

In [ ]:
# Complete cross-section
concrete = get_concrete_by_class('C30/37')
shape = RectangularShape(b=300, h=500)

# Wrap layers in ReinforcementLayout
reinforcement = ReinforcementLayout(layers=[bottom_layer, top_layer])

cs_bars = CrossSection(
    shape=shape,
    concrete=concrete.matmod,
    reinforcement=reinforcement
)

print("Cross-Section with BarReinforcement:")
print("=" * 60)
print(f"Geometry: {shape.b} × {shape.h} mm")
print(f"Concrete: {concrete.name} (f_cd = {concrete.f_cd:.2f} MPa)")
print(f"\nReinforcement:")
print(f"  Bottom: {bottom_layer.count}×D{bottom_layer.diameter} ({bottom_layer.component.product_id})")
print(f"  Top:    {top_layer.count}×D{top_layer.diameter} ({top_layer.component.product_id})")
print(f"  Total:  {bottom_layer.A_s + top_layer.A_s:.0f} mm²")
print(f"  Ratio:  {(bottom_layer.A_s + top_layer.A_s) / shape.area * 100:.2f}%")

In [ ]:
# Visualize
cs_bars.plot_cross_section()
plt.title(f'Rectangular Beam with BarReinforcement\n' +
          f'{concrete.name}, {bottom_layer.count}×D{bottom_layer.diameter} + {top_layer.count}×D{top_layer.diameter}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Example 3.3: Carbon Bars (Different Material)

In [ ]:
# Carbon bar component
carbon_d8 = CarbonBarComponent(nominal_diameter=8, grade='C1570')

print(f"Carbon Bar Component: {carbon_d8.name}")
print(f"  Product ID: {carbon_d8.product_id}")
print(f"  Diameter: {carbon_d8.nominal_diameter} mm")
print(f"  Area: {carbon_d8.area:.2f} mm²")
print(f"  f_tk: {carbon_d8.f_tk} MPa")
print(f"  f_td: {carbon_d8.f_td:.1f} MPa")
print(f"  E: {carbon_d8.E} MPa")

# Create layer
carbon_layer = BarReinforcement(
    z=450,
    component=carbon_d8,
    count=8,
    name='Carbon bottom'
)

print(f"\nCarbon BarReinforcement:")
print(f"  {carbon_layer.count} × D{carbon_layer.diameter}")
print(f"  A_s = {carbon_layer.A_s:.2f} mm²")

## 4. LayerReinforcement - Textile/Mat Proxy

**Usage:** Product-based analysis with textile reinforcement.

**Pattern:**
- Link to textile/mat component
- Specify width
- `A_s` calculated automatically: `(width / spacing) × A_roving`

In [ ]:
# Create textile component
carbon_textile = TextileReinforcementComponent(
    product_id='CARB-TEX-001',
    name='Carbon Textile 50k',
    manufacturer='Example Corp',
    material_type='carbon',
    spacing=14,        # mm between rovings
    A_roving=1.8,      # mm² per roving
    f_tk=2500,         # MPa
    E=200000,          # MPa
    eps_uk=0.015
)

print("Textile Component:")
print("=" * 60)
print(f"  Product: {carbon_textile.name}")
print(f"  Product ID: {carbon_textile.product_id}")
print(f"  Spacing: {carbon_textile.spacing} mm")
print(f"  A_roving: {carbon_textile.A_roving} mm²")
print(f"  f_tk: {carbon_textile.f_tk} MPa")
print(f"  E: {carbon_textile.E} MPa")

In [ ]:
# Create LayerReinforcement (proxy)
textile_layer = LayerReinforcement(
    z=25,
    component=carbon_textile,  # Link to catalog
    width=300,                 # Active width
    name='Textile layer'
)

print("LayerReinforcement:")
print("=" * 60)
print(f"  Position: z = {textile_layer.z} mm")
print(f"  Component: {textile_layer.component.product_id}")
print(f"  Width: {textile_layer.width} mm")
print(f"  Spacing: {textile_layer.component.spacing} mm")
print(f"  A_roving: {textile_layer.component.A_roving} mm²")
print(f"  Number of rovings: {textile_layer.n_rovings}")
print(f"  A_s = ({textile_layer.width}/{textile_layer.component.spacing}) × {textile_layer.component.A_roving}")
print(f"      = {textile_layer.n_rovings} × {textile_layer.component.A_roving}")
print(f"      = {textile_layer.A_s:.2f} mm²")

### Example 4.1: T-Beam with Textile + Steel

Hybrid reinforcement with cached component lookups.

In [ ]:
# T-beam with textile top layer and steel bottom layer
t_shape = TShape(b_f=800, h_f=150, b_w=300, h=600)
concrete_c40 = get_concrete_by_class('C40/50')

# Steel bottom
steel_d25 = SteelRebarComponent(nominal_diameter=25, grade='B500B')
bottom_steel = BarReinforcement(z=550, component=steel_d25, count=6)

# Textile top
textile_top = LayerReinforcement(z=25, component=carbon_textile, width=600)

# Wrap layers in ReinforcementLayout
reinforcement_hybrid = ReinforcementLayout(layers=[bottom_steel, textile_top])

cs_hybrid = CrossSection(
    shape=t_shape,
    concrete=concrete_c40.matmod,
    reinforcement=reinforcement_hybrid
)

print("Hybrid Cross-Section (Steel + Textile):")
print("=" * 60)
print(f"Shape: T-beam (flange {t_shape.b_f}×{t_shape.h_f}, web {t_shape.b_w}×{t_shape.h-t_shape.h_f})")
print(f"Concrete: {concrete_c40.name}")
print(f"\nBottom reinforcement (BarReinforcement):")
print(f"  {bottom_steel.count}×D{bottom_steel.diameter} steel bars")
print(f"  A_s = {bottom_steel.A_s:.0f} mm²")
print(f"\nTop reinforcement (LayerReinforcement):")
print(f"  Carbon textile, width = {textile_top.width} mm")
print(f"  {textile_top.n_rovings} rovings")
print(f"  A_s = {textile_top.A_s:.2f} mm²")

In [ ]:
# Visualize hybrid cross-section
cs_hybrid.plot_cross_section()
plt.title(f'Hybrid T-Beam: Steel Bars + Carbon Textile\n' +
          f'{concrete_c40.name}, {bottom_steel.count}×D{bottom_steel.diameter} + Textile',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. AreaReinforcement - Product-Independent

**Usage:** Design-oriented workflow where area is determined by calculations.

**Pattern:**
- No catalog link
- Specify `A_s` and `material` explicitly
- Used for optimization/design equations

In [ ]:
# Design scenario: required area from design equations
A_s_required_bottom = 2500  # mm² from ultimate moment design
A_s_required_top = 800      # mm² for crack control

# Create AreaReinforcement layers
area_bottom = AreaReinforcement(
    z=450,
    A_s=A_s_required_bottom,
    material=create_steel('B500B'),  # Material model directly
    name='Bottom (design)'
)

area_top = AreaReinforcement(
    z=50,
    A_s=A_s_required_top,
    material=create_steel('B500B'),
    name='Top (design)'
)

print("AreaReinforcement Layers:")
print("=" * 60)
print(f"\nBottom layer:")
print(f"  Position: z = {area_bottom.z} mm")
print(f"  Required A_s = {area_bottom.A_s} mm²")
print(f"  Material: {type(area_bottom.material).__name__}")
print(f"  No catalog component (product-independent)")

print(f"\nTop layer:")
print(f"  Position: z = {area_top.z} mm")
print(f"  Required A_s = {area_top.A_s} mm²")
print(f"  Material: {type(area_top.material).__name__}")

print(f"\nTotal required: {area_bottom.A_s + area_top.A_s} mm²")

In [ ]:
# Cross-section with AreaReinforcement
shape_design = RectangularShape(b=300, h=500)
concrete_design = get_concrete_by_class('C30/37')

# Wrap layers in ReinforcementLayout
reinforcement_area = ReinforcementLayout(layers=[area_bottom, area_top])

cs_area = CrossSection(
    shape=shape_design,
    concrete=concrete_design.matmod,
    reinforcement=reinforcement_area
)

print("Cross-Section with AreaReinforcement:")
print("=" * 60)
print(f"Geometry: {shape_design.b} × {shape_design.h} mm")
print(f"Concrete: {concrete_design.name}")
print(f"\nReinforcement (design values):")
print(f"  Bottom: A_s = {area_bottom.A_s} mm²")
print(f"  Top:    A_s = {area_top.A_s} mm²")
print(f"  Ratio:  {(area_bottom.A_s + area_top.A_s) / shape_design.area * 100:.2f}%")
print(f"\nNote: Product selection pending - need to select bars to match areas")

In [ ]:
# Visualize
cs_area.plot_cross_section()
plt.title(f'Cross-Section with AreaReinforcement (Design Mode)\n' +
          f'Bottom: {area_bottom.A_s} mm², Top: {area_top.A_s} mm²',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Example 5.1: Component Selection for AreaReinforcement

After design calculations determine required areas, select actual components from **cached catalog**.

In [ ]:
# Load steel catalog (uses cache automatically - very fast)
steel_catalog = create_steel_rebar_catalog()

print("Component Selection for Bottom Layer:")
print("=" * 70)
print(f"Required: A_s = {A_s_required_bottom} mm²")
print(f"Catalog entries: {len(steel_catalog)} (loaded from cache)\n")

# Find suitable combinations
options = []
for _, bar in steel_catalog.iterrows():
    for count in range(3, 12):  # Try 3-11 bars
        total_area = count * bar['area']
        if 0.95 * A_s_required_bottom <= total_area <= 1.10 * A_s_required_bottom:
            options.append({
                'diameter': bar['nominal_diameter'],
                'count': count,
                'area_per_bar': bar['area'],
                'total_area': total_area,
                'product_id': bar['product_id'],
                'deviation': (total_area - A_s_required_bottom) / A_s_required_bottom * 100
            })

# Sort by proximity to required
options_sorted = sorted(options, key=lambda x: abs(x['deviation']))

print("Best options (within ±10%):")
print("-" * 70)
for i, opt in enumerate(options_sorted[:5], 1):
    print(f"{i}. {opt['count']}×D{opt['diameter']} = {opt['total_area']:.0f} mm²  ({opt['deviation']:+.1f}%)")
    print(f"   Product: {opt['product_id']}")

In [ ]:
# Select best option and create BarReinforcement
best_option = options_sorted[0]
selected_steel = SteelRebarComponent(
    nominal_diameter=best_option['diameter'],
    grade='B500B'
)

# Replace AreaReinforcement with BarReinforcement
finalized_bottom = BarReinforcement(
    z=450,
    component=selected_steel,
    count=best_option['count'],
    name='Bottom (finalized)'
)

print("\nFinalized Bottom Layer:")
print("=" * 60)
print(f"  Changed from: AreaReinforcement (A_s = {A_s_required_bottom} mm²)")
print(f"  To: BarReinforcement")
print(f"      Component: {finalized_bottom.component.product_id}")
print(f"      Configuration: {finalized_bottom.count}×D{finalized_bottom.diameter}")
print(f"      Actual A_s: {finalized_bottom.A_s:.0f} mm²")
print(f"      Deviation: {(finalized_bottom.A_s - A_s_required_bottom)/A_s_required_bottom*100:+.1f}%")

## 6. Comparison: All Three Types

Summary of reinforcement layer types and their use cases.

In [ ]:
# Create comparison table
comparison = pd.DataFrame([
    {
        'Type': 'BarReinforcement',
        'Catalog Link': '✓ Yes (bars)',
        'Area Calculation': 'component.area × count',
        'Material Source': 'component.matmod',
        'Use Case': 'Product-based analysis',
        'Traceability': '✓ Full',
        'Example': '6×D20 steel bars'
    },
    {
        'Type': 'LayerReinforcement',
        'Catalog Link': '✓ Yes (textiles)',
        'Area Calculation': '(width/spacing) × A_roving',
        'Material Source': 'component.matmod',
        'Use Case': 'Product-based analysis',
        'Traceability': '✓ Full',
        'Example': 'Carbon textile 300mm width'
    },
    {
        'Type': 'AreaReinforcement',
        'Catalog Link': '✗ No',
        'Area Calculation': 'Explicit A_s',
        'Material Source': 'Explicit material',
        'Use Case': 'Design optimization',
        'Traceability': '✗ Pending',
        'Example': 'A_s = 2500 mm² (TBD)'
    }
])

print("\nReinforcement Layer Type Comparison:")
print("=" * 100)
print(comparison.to_string(index=False))
print("=" * 100)

## 7. Workflow Example: Design → Analysis

Complete workflow showing both design-oriented and product-based approaches with cached catalogs.

In [ ]:
print("WORKFLOW: Design → Analysis")
print("=" * 80)

# Step 1: Design phase (AreaReinforcement)
print("\n[1] DESIGN PHASE - Calculate required areas")
print("-" * 80)
design_layers = [
    AreaReinforcement(z=450, A_s=2500, material=create_steel('B500B'), name='Bottom design'),
    AreaReinforcement(z=50, A_s=800, material=create_steel('B500B'), name='Top design')
]

for layer in design_layers:
    print(f"  {layer.name}: A_s = {layer.A_s} mm² (from design equations)")

# Step 2: Component selection
print("\n[2] COMPONENT SELECTION - Choose products from catalog")
print("-" * 80)
print("  Bottom: 8×D20 = 2513 mm² (selected from steel catalog)")
print("  Top:    4×D16 = 804 mm² (selected from steel catalog)")

# Step 3: Analysis phase (BarReinforcement)
print("\n[3] ANALYSIS PHASE - Product-based with traceability")
print("-" * 80)
steel_d20_final = SteelRebarComponent(nominal_diameter=20, grade='B500B')
steel_d16_final = SteelRebarComponent(nominal_diameter=16, grade='B500B')

analysis_layers = [
    BarReinforcement(z=450, component=steel_d20_final, count=8, name='Bottom analysis'),
    BarReinforcement(z=50, component=steel_d16_final, count=4, name='Top analysis')
]

for layer in analysis_layers:
    print(f"  {layer.name}: {layer.count}×D{layer.diameter}")
    print(f"    Product: {layer.component.product_id}")
    print(f"    A_s = {layer.A_s:.0f} mm²")

# Step 4: Final cross-section
print("\n[4] FINAL CROSS-SECTION - Ready for moment-curvature analysis")
print("-" * 80)
concrete_final = get_concrete_by_class('C30/37')
shape_final = RectangularShape(b=300, h=500)

# Wrap layers in ReinforcementLayout
reinforcement_final = ReinforcementLayout(layers=analysis_layers)

cs_final = CrossSection(
    shape=shape_final,
    concrete=concrete_final.matmod,
    reinforcement=reinforcement_final
)

print(f"  Geometry: {shape_final.b}×{shape_final.h} mm")
print(f"  Concrete: {concrete_final.product_id}")
print(f"  Reinforcement: Full product traceability")
print(f"  Status: ✓ Ready for mkappa analysis")

In [ ]:
# Visualize final design
cs_final.plot_cross_section()
plt.title(f'Final Design: {shape_final.b}×{shape_final.h} mm\n' +
          f'{concrete_final.name}, ' +
          f'{analysis_layers[0].count}×D{analysis_layers[0].diameter} + ' +
          f'{analysis_layers[1].count}×D{analysis_layers[1].diameter}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Summary

### Three Reinforcement Layer Types:

1. **BarReinforcement**
   - Proxy to bar catalog (steel, carbon)
   - `A_s = component.area × count`
   - Full product traceability
   - Use: Product-based analysis

2. **LayerReinforcement**
   - Proxy to textile/mat catalog
   - `A_s = (width/spacing) × A_roving`
   - Full product traceability
   - Use: Distributed reinforcement analysis

3. **AreaReinforcement**
   - No catalog link
   - Explicit `A_s` and `material`
   - Product selection pending
   - Use: Design optimization

### Benefits:
- ✓ Flexible workflow (design ↔ analysis)
- ✓ Product traceability when needed
- ✓ Product-independent when optimizing
- ✓ Lightweight proxy pattern (no duplication)
- ✓ **Automatic catalog caching** for performance (NEW)
- ✓ Ready for Phase 3 (mkappa analysis)

### Caching System:
- ✓ Transparent: No code changes needed
- ✓ Fast: 2-10x speedup after first load
- ✓ Persistent: JSON storage survives sessions
- ✓ See [12_catalog_caching_demo.ipynb](12_catalog_caching_demo.ipynb) for details

In [ ]:
print("✓ Notebook complete: Reinforcement layer types with catalog integration")
print("  - BarReinforcement: Discrete bar proxy")
print("  - LayerReinforcement: Textile/mat proxy")
print("  - AreaReinforcement: Product-independent")
print("  - Complete design → analysis workflow")
print("  - Automatic catalog caching for performance")
print("\nNote: All catalog operations automatically use caching")
print("      See 12_catalog_caching_demo.ipynb for caching details")